# Step 1 — Geographic Concentration & Market Dependency
**Objective:** Understand who Brazil's top trading partners are and how those relationships have evolved over 30 years.

This notebook follows a macro-to-micro structure:
1. Total trade value by year (exports vs imports)
2. Top trading partners nationally over time
3. Structural shifts — key moments and transitions
4. Share of trade by regional bloc over time

## Setup

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os
from pathlib import Path
from sqlalchemy import create_engine
from dotenv import load_dotenv

# Credentials
dotenv_path = Path(r"C:\Users\e_koh\Downloads\State Analysis\brazil-state-trade-analysis\.env")
load_dotenv(dotenv_path, override=True)

DB_USER     = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST     = os.getenv("DB_HOST")
DB_PORT     = os.getenv("DB_PORT")
DB_NAME     = os.getenv("DB_NAME")

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")
print("Connected to database successfully")

## Data is capped at 2025 — 2026 contains only partial year data which distorts trend visuals
MAX_YEAR = 2025

Connected to database successfully


## 1.1 — Total Trade Value by Year (Exports vs Imports)
The broadest possible view: how has Brazil's total trade volume evolved since 1997?

In [ ]:
query_total = f"""
    SELECT "CO_ANO" AS year,
           SUM("VL_FOB") AS total_fob
    FROM exp
    WHERE "CO_ANO" <= {MAX_YEAR}
    GROUP BY "CO_ANO"
    ORDER BY "CO_ANO"
"""

query_total_imp = f"""
    SELECT "CO_ANO" AS year,
           SUM("VL_FOB") AS total_fob
    FROM imp
    WHERE "CO_ANO" <= {MAX_YEAR}
    GROUP BY "CO_ANO"
    ORDER BY "CO_ANO"
"""

df_exp_total = pd.read_sql(query_total, engine)
df_imp_total = pd.read_sql(query_total_imp, engine)

# Convert to USD billions
df_exp_total['total_usd_bn'] = df_exp_total['total_fob'] / 1e9
df_imp_total['total_usd_bn'] = df_imp_total['total_fob'] / 1e9

# Plot
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df_exp_total['year'], df_exp_total['total_usd_bn'], label='Exports', color='steelblue', linewidth=2)
ax.plot(df_imp_total['year'], df_imp_total['total_usd_bn'], label='Imports', color='tomato', linewidth=2)
ax.fill_between(df_exp_total['year'], df_exp_total['total_usd_bn'], df_imp_total['total_usd_bn'],
                where=df_exp_total['total_usd_bn'] >= df_imp_total['total_usd_bn'],
                alpha=0.15, color='steelblue', label='Trade Surplus')
ax.fill_between(df_exp_total['year'], df_exp_total['total_usd_bn'], df_imp_total['total_usd_bn'],
                where=df_exp_total['total_usd_bn'] < df_imp_total['total_usd_bn'],
                alpha=0.15, color='tomato', label='Trade Deficit')

# Annotate key events
events = {2008: '2008\nFinancial Crisis', 2014: '2014\nRecession', 2020: 'COVID-19'}
for year, label in events.items():
    ax.axvline(x=year, color='gray', linestyle='--', linewidth=1, alpha=0.7)
    ax.text(year + 0.2, ax.get_ylim()[1] * 0.85, label, fontsize=8, color='gray')

ax.set_title(f"Brazil Total Trade Value (1997–{MAX_YEAR})", fontsize=14, fontweight='bold')
ax.set_xlabel("Year")
ax.set_ylabel("Trade Value (USD Billions)")
ax.legend()
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0fbn'))
plt.tight_layout()
plt.savefig('output_1_1_total_trade.png', dpi=150)
plt.show()
print(df_exp_total.tail())

## 1.2 — Top 10 Export Partners by Year
Which countries have received the most Brazilian exports, and how has that ranking evolved?

In [ ]:
query_top_partners = f"""
    SELECT e."CO_ANO" AS year,
           p.nome_pais_ing AS country,
           SUM(e."VL_FOB") AS total_fob
    FROM exp e
    JOIN pais p ON e."CO_PAIS" = p.codigo_pais
    WHERE e."CO_ANO" <= {MAX_YEAR}
    GROUP BY e."CO_ANO", p.nome_pais_ing
    ORDER BY e."CO_ANO", total_fob DESC
"""

df_partners = pd.read_sql(query_top_partners, engine)
df_partners['total_usd_bn'] = df_partners['total_fob'] / 1e9

# Get top 10 countries by total exports across all years
top10_countries = (
    df_partners.groupby('country')['total_fob']
    .sum()
    .nlargest(10)
    .index.tolist()
)

df_top10 = df_partners[df_partners['country'].isin(top10_countries)]

# Pivot for plotting
df_pivot = df_top10.pivot(index='year', columns='country', values='total_usd_bn').fillna(0)

# Plot
fig, ax = plt.subplots(figsize=(14, 7))
for country in df_pivot.columns:
    ax.plot(df_pivot.index, df_pivot[country], label=country, linewidth=2)

ax.set_title(f"Brazil Top 10 Export Partners (1997–{MAX_YEAR})", fontsize=14, fontweight='bold')
ax.set_xlabel("Year")
ax.set_ylabel("Export Value (USD Billions)")
ax.legend(loc='upper left', fontsize=8)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0fbn'))
plt.tight_layout()
plt.savefig('output_1_2_top10_export_partners.png', dpi=150)
plt.show()

## 1.3 — China vs USA: The Structural Shift
One of the most significant structural changes in Brazilian trade over the last 30 years is China overtaking the United States as Brazil's largest trading partner. This cell isolates that story.

In [ ]:
query_china_usa = f"""
    SELECT e."CO_ANO" AS year,
           p.nome_pais_ing AS country,
           SUM(e."VL_FOB") AS total_fob
    FROM exp e
    JOIN pais p ON e."CO_PAIS" = p.codigo_pais
    WHERE p.nome_pais_ing IN ('China', 'United States')
    AND e."CO_ANO" <= {MAX_YEAR}
    GROUP BY e."CO_ANO", p.nome_pais_ing
    ORDER BY e."CO_ANO"
"""

df_china_usa = pd.read_sql(query_china_usa, engine)
df_china_usa['total_usd_bn'] = df_china_usa['total_fob'] / 1e9
df_pivot_cu = df_china_usa.pivot(index='year', columns='country', values='total_usd_bn').fillna(0)

fig, ax = plt.subplots(figsize=(14, 6))
for country in df_pivot_cu.columns:
    ax.plot(df_pivot_cu.index, df_pivot_cu[country], label=country, linewidth=2.5)

# Find and annotate the crossover year
cols = df_pivot_cu.columns.tolist()
if len(cols) == 2:
    crossover = df_pivot_cu[df_pivot_cu[cols[0]] > df_pivot_cu[cols[1]]].index
    if len(crossover) > 0:
        ax.axvline(x=crossover[0], color='gray', linestyle='--', linewidth=1)
        ax.text(crossover[0] + 0.2, df_pivot_cu.max().max() * 0.8,
                f'Crossover\n{crossover[0]}', fontsize=9, color='gray')

ax.set_title(f"Brazil Exports: China vs United States (1997–{MAX_YEAR})", fontsize=14, fontweight='bold')
ax.set_xlabel("Year")
ax.set_ylabel("Export Value (USD Billions)")
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0fbn'))
plt.tight_layout()
plt.savefig('output_1_3_china_vs_usa.png', dpi=150)
plt.show()

## 1.4 — Share of Trade by Regional Bloc Over Time
How has the distribution of Brazilian trade across regional blocs (Mercosul, EU, Asia, etc.) shifted over 30 years?

In [ ]:
query_blocs = f"""
    SELECT e."CO_ANO" AS year,
           pb.nome_bloco_ing AS bloc,
           SUM(e."VL_FOB") AS total_fob
    FROM exp e
    JOIN pais_bloco pb ON e."CO_PAIS" = pb.codigo_pais
    WHERE e."CO_ANO" <= {MAX_YEAR}
    GROUP BY e."CO_ANO", pb.nome_bloco_ing
    ORDER BY e."CO_ANO", total_fob DESC
"""

query_blocs_imp = f"""
    SELECT i."CO_ANO" AS year,
           pb.nome_bloco_ing AS bloc,
           SUM(i."VL_FOB") AS total_fob
    FROM imp i
    JOIN pais_bloco pb ON i."CO_PAIS" = pb.codigo_pais
    WHERE i."CO_ANO" <= {MAX_YEAR}
    GROUP BY i."CO_ANO", pb.nome_bloco_ing
    ORDER BY i."CO_ANO", total_fob DESC
"""

df_blocs     = pd.read_sql(query_blocs, engine)
df_blocs_imp = pd.read_sql(query_blocs_imp, engine)

def build_bloc_pct(df, n=6):
    df['total_usd_bn'] = df['total_fob'] / 1e9
    top_blocs = (
        df.groupby('bloc')['total_fob']
        .sum()
        .nlargest(n)
        .index.tolist()
    )
    df_top    = df[df['bloc'].isin(top_blocs)]
    df_pivot  = df_top.pivot(index='year', columns='bloc', values='total_usd_bn').fillna(0)
    df_pct    = df_pivot.div(df_pivot.sum(axis=1), axis=0) * 100
    return df_pct

df_exp_pct = build_bloc_pct(df_blocs)
df_imp_pct = build_bloc_pct(df_blocs_imp)

def add_pct_annotations(ax, df_pct, start_year, end_year):
    ## Annotate the share of each bloc in start_year and end_year on the right side
    for bloc in df_pct.columns:
        start_val = df_pct.loc[start_year, bloc] if start_year in df_pct.index else 0
        end_val   = df_pct.loc[end_year,   bloc] if end_year   in df_pct.index else 0
        if end_val > 1:  ## Only annotate blocs with meaningful share
            ax.annotate(
                f"{bloc}: {start_val:.1f}% → {end_val:.1f}%",
                xy=(end_year, df_pct.loc[:end_year, bloc].cumsum().iloc[-1] - end_val / 2),
                xytext=(end_year + 0.3, df_pct.loc[:end_year, bloc].cumsum().iloc[-1] - end_val / 2),
                fontsize=7, va='center'
            )

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 14))

## ---- EXPORTS ----
ax1.stackplot(df_exp_pct.index, df_exp_pct.T, labels=df_exp_pct.columns, alpha=0.8)
ax1.set_title(f"Brazil Export Share by Regional Bloc (1997–{MAX_YEAR})", fontsize=14, fontweight='bold')
ax1.set_xlabel("Year")
ax1.set_ylabel("Share of Total Exports (%)")
ax1.legend(loc='upper left', fontsize=8)
ax1.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))

## Add 1997 and MAX_YEAR percentage summary below the export chart
start_year, end_year = df_exp_pct.index[0], df_exp_pct.index[-1]
summary_exp = pd.DataFrame({
    'Bloc'            : df_exp_pct.columns,
    f'{start_year} (%)': df_exp_pct.loc[start_year].round(1).values,
    f'{end_year} (%)' : df_exp_pct.loc[end_year].round(1).values,
    'Change (pp)'     : (df_exp_pct.loc[end_year] - df_exp_pct.loc[start_year]).round(1).values
})

## ---- IMPORTS ----
ax2.stackplot(df_imp_pct.index, df_imp_pct.T, labels=df_imp_pct.columns, alpha=0.8)
ax2.set_title(f"Brazil Import Share by Regional Bloc (1997–{MAX_YEAR})", fontsize=14, fontweight='bold')
ax2.set_xlabel("Year")
ax2.set_ylabel("Share of Total Imports (%)")
ax2.legend(loc='upper left', fontsize=8)
ax2.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))

## Add 1997 and MAX_YEAR percentage summary below the import chart
summary_imp = pd.DataFrame({
    'Bloc'            : df_imp_pct.columns,
    f'{start_year} (%)': df_imp_pct.loc[start_year].round(1).values,
    f'{end_year} (%)' : df_imp_pct.loc[end_year].round(1).values,
    'Change (pp)'     : (df_imp_pct.loc[end_year] - df_imp_pct.loc[start_year]).round(1).values
})

plt.tight_layout()
plt.savefig('output_1_4_bloc_share.png', dpi=150)
plt.show()

## Summary tables showing share in first and last year and the change in percentage points
print(f"Export bloc share — {start_year} vs {end_year}")
print(summary_exp.sort_values(f'{end_year} (%)', ascending=False).to_string(index=False))
print(f"\nImport bloc share — {start_year} vs {end_year}")
print(summary_imp.sort_values(f'{end_year} (%)', ascending=False).to_string(index=False))

## 1.5 — Summary Table: Top 10 Partners (Most Recent Year)
A clean summary of the current state — who are Brazil's top 10 export and import partners today?

In [ ]:
## 2026 data excluded — only partial year available, which distorts the summary figures
## Using MAX_YEAR = 2025 as the reference year for the summary table

# Query total exports and imports for MAX_YEAR (all countries)
query_total_exp_year = f"SELECT SUM(\"VL_FOB\") AS total FROM exp WHERE \"CO_ANO\" = {MAX_YEAR}"
query_total_imp_year = f"SELECT SUM(\"VL_FOB\") AS total FROM imp WHERE \"CO_ANO\" = {MAX_YEAR}"
total_exp = pd.read_sql(query_total_exp_year, engine)['total'].iloc[0]
total_imp = pd.read_sql(query_total_imp_year, engine)['total'].iloc[0]

query_latest_exp = f"""
    SELECT p.nome_pais_ing AS country,
           SUM(e."VL_FOB") AS exports_usd
    FROM exp e
    JOIN pais p ON e."CO_PAIS" = p.codigo_pais
    WHERE e."CO_ANO" = {MAX_YEAR}
    GROUP BY p.nome_pais_ing
    ORDER BY exports_usd DESC
    LIMIT 10
"""

query_latest_imp = f"""
    SELECT p.nome_pais_ing AS country,
           SUM(i."VL_FOB") AS imports_usd
    FROM imp i
    JOIN pais p ON i."CO_PAIS" = p.codigo_pais
    WHERE i."CO_ANO" = {MAX_YEAR}
    GROUP BY p.nome_pais_ing
    ORDER BY imports_usd DESC
    LIMIT 10
"""

df_latest_exp = pd.read_sql(query_latest_exp, engine)
df_latest_imp = pd.read_sql(query_latest_imp, engine)

df_latest_exp['exports_usd_bn'] = (df_latest_exp['exports_usd'] / 1e9).round(2)
df_latest_exp['share_%'] = (df_latest_exp['exports_usd'] / total_exp * 100).round(2)

df_latest_imp['imports_usd_bn'] = (df_latest_imp['imports_usd'] / 1e9).round(2)
df_latest_imp['share_%'] = (df_latest_imp['imports_usd'] / total_imp * 100).round(2)

print(f"Top 10 Export Partners ({MAX_YEAR}) — Total Exports: USD {total_exp/1e9:.1f}bn")
print(df_latest_exp[['country', 'exports_usd_bn', 'share_%']].to_string(index=False))
print(f"\nTop 10 Import Partners ({MAX_YEAR}) — Total Imports: USD {total_imp/1e9:.1f}bn")
print(df_latest_imp[['country', 'imports_usd_bn', 'share_%']].to_string(index=False))

## 1.6 — Trade Balance by Country (Largest Surpluses & Deficits)
Which countries does Brazil run its largest trade surpluses and deficits with? A positive balance means Brazil exports more than it imports from that country. A negative balance means the opposite.

In [2]:
## Trade balance = exports - imports per country for MAX_YEAR
## Positive = surplus (Brazil exports more to that country than it imports)
## Negative = deficit (Brazil imports more from that country than it exports)
## Using subqueries to pre-filter by year to avoid PostgreSQL type conflict in JOIN conditions

query_balance = f"""
    SELECT p.nome_pais_ing AS country,
           COALESCE(e.exports_usd, 0) AS exports_usd,
           COALESCE(i.imports_usd, 0) AS imports_usd,
           COALESCE(e.exports_usd, 0) - COALESCE(i.imports_usd, 0) AS balance_usd
    FROM pais p
    LEFT JOIN (
        SELECT "CO_PAIS", SUM("VL_FOB") AS exports_usd
        FROM exp
        WHERE "CO_ANO" = {MAX_YEAR}
        GROUP BY "CO_PAIS"
    ) e ON e."CO_PAIS" = p.codigo_pais
    LEFT JOIN (
        SELECT "CO_PAIS", SUM("VL_FOB") AS imports_usd
        FROM imp
        WHERE "CO_ANO" = {MAX_YEAR}
        GROUP BY "CO_PAIS"
    ) i ON i."CO_PAIS" = p.codigo_pais
    WHERE COALESCE(e.exports_usd, 0) > 0 OR COALESCE(i.imports_usd, 0) > 0
    ORDER BY balance_usd DESC
"""

df_balance = pd.read_sql(query_balance, engine)
df_balance['exports_usd_bn'] = (df_balance['exports_usd'] / 1e9).round(2)
df_balance['imports_usd_bn'] = (df_balance['imports_usd'] / 1e9).round(2)
df_balance['balance_usd_bn'] = (df_balance['balance_usd'] / 1e9).round(2)

## Top 10 surpluses — countries where Brazil exports most relative to imports
df_surplus = df_balance.head(10)[['country', 'exports_usd_bn', 'imports_usd_bn', 'balance_usd_bn']]

## Top 10 deficits — countries where Brazil imports most relative to exports
df_deficit = df_balance.tail(10).sort_values('balance_usd_bn')[['country', 'exports_usd_bn', 'imports_usd_bn', 'balance_usd_bn']]

print(f"10 Largest Trade Surpluses ({MAX_YEAR}) — countries Brazil exports most to relative to imports")
print(df_surplus.to_string(index=False))
print(f"\n10 Largest Trade Deficits ({MAX_YEAR}) — countries Brazil imports most from relative to exports")
print(df_deficit.to_string(index=False))

10 Largest Trade Surpluses (2025) — countries Brazil exports most to relative to imports
             country  exports_usd_bn  imports_usd_bn  balance_usd_bn
               China           99.94           70.92           29.02
         Netherlands           11.73            2.37            9.36
           Argentina           18.11           12.93            5.17
               Spain            8.78            3.82            4.96
              Canada            7.25            3.14            4.11
           Singapore            7.35            3.30            4.05
United Arab Emirates            3.76            0.63            3.14
                Iran            2.92            0.08            2.84
              Turkey            4.15            1.42            2.73
               Chile            7.18            4.63            2.55

10 Largest Trade Deficits (2025) — countries Brazil imports most from relative to exports
      country  exports_usd_bn  imports_usd_bn  balance_usd_bn